In [ ]:
# --- Import libraries for data handling, date offsets, file paths, statistics ---
import pandas as pd
from pandas.tseries.offsets import BDay  # business-day window selection
from pathlib import Path
from math import sqrt
from scipy import stats  # two-sided t distribution for significance tests

In [ ]:
# --- Model calibration and event definition ---
alpha = -0.0002  # intercept from the CAPM regression
beta = 1.779     # market beta estimated over the estimation window
event_date = pd.Timestamp("2020-03-11")  # WHO pandemic announcement (Norwegian Air focus date)


In [ ]:
# Kursdata Norwegian
nas = (pd.read_csv(Path("data/norway") / "norwegian.csv", skiprows=[1, 2])
       .rename(columns={"Price": "Date"})
       .assign(Date=lambda df: pd.to_datetime(df["Date"]),
               Close=lambda df: pd.to_numeric(df["Close"], errors="coerce"))
       .dropna(subset=["Close"])
       .set_index("Date")
       .sort_index())

# OSEBX og risikofri rente
osebx = (pd.read_csv(Path("data/norway") / "osebx_index.csv", skiprows=[1, 2])
         .rename(columns={"Price": "Date"})
         .assign(Date=lambda df: pd.to_datetime(df["Date"]),
                 Close=lambda df: pd.to_numeric(df["Close"], errors="coerce"))
         .dropna(subset=["Close"])
         .set_index("Date")
         .sort_index())

rf = (pd.read_csv(Path("data/norway") / "Norway_Rf_daily.csv", skiprows=1)
      .rename(columns={"date": "Date", "Rf(1d)": "rf"})
      .assign(Date=lambda df: pd.to_datetime(df["Date"], format="%Y%m%d"),
              rf=lambda df: pd.to_numeric(df["rf"], errors="coerce"))
      .dropna(subset=["rf"])
      .set_index("Date")
      .sort_index())

# Daglige avkastninger
returns = (pd.concat([
    nas["Close"].pct_change().rename("norwegian_return"),
    osebx["Close"].pct_change().rename("osebx_return"),
    rf["rf"]
], axis=1).dropna())

In [ ]:


summary = []
for k in [15,10, 5, 3, 1]:
    win = returns.loc[event_date - BDay(k): event_date + BDay(k)].copy()
    win["norwegian_excess"] = win["norwegian_return"] - win["rf"]
    win["osebx_excess"] = win["osebx_return"] - win["rf"]
    win["expected_capm"] = alpha + beta * win["osebx_excess"]
    ar = win["norwegian_excess"] - win["expected_capm"]

    n = len(ar)
    mean_ar = ar.mean()
    std_ar = ar.std(ddof=1)
    t_stat = mean_ar / (std_ar / sqrt(n)) if std_ar > 0 else float("nan")
    p_val = 2 * stats.t.sf(abs(t_stat), df=n - 1) if std_ar > 0 else float("nan")

    summary.append({
        "window": f"[-{k}, +{k}]",
        "n": n,
        "mean_AR": mean_ar,
        "std_AR": std_ar,
        "CAR": ar.sum(),
        "t_stat": t_stat,
        "p_value": p_val
    })

results = pd.DataFrame(summary)
results